# Student Lab: Code Smell Detection Với DeepSmells

## Mục tiêu học tập

Sau khi hoàn thành lab này, sinh viên có thể:

1. Nạp bộ dữ liệu code smell đã được tokenized từ Kaggle Dataset.
2. Tự viết các bước chính trong pipeline dữ liệu: tìm tokenizer_cs, đếm dữ liệu, padding, chia train/validation.
3. Tự cài đặt PyTorch Dataset và kiểm tra shape batch.
4. Tự cài đặt kiến trúc DeepSmells: 1D-CNN -> LSTM/BiLSTM -> classifier.
5. Viết metric đánh giá cho dữ liệu mất cân bằng.
6. Chạy forward pass và, nếu muốn, chạy training ngắn trên Kaggle.

## Ví dụ mở đầu: một code smell trong thực tế

Trước khi đi vào mô hình, ta xem một ví dụ đơn giản về **Complex Method**. Đây là một method làm quá nhiều việc cùng lúc: kiểm tra trạng thái user, kiểm tra giỏ hàng, tính giảm giá, tính phí ship, tính thuế và trả về kết quả cuối cùng.

~~~java
public double calculateFinalPrice(User user, Cart cart) {
    if (user != null && user.isActive()) {
        if (cart != null && !cart.getItems().isEmpty()) {
            double total = 0;
            for (Item item : cart.getItems()) {
                if (item.isAvailable()) {
                    if (item.getCategory().equals("BOOK")) {
                        total += item.getPrice() * 0.9;
                    } else if (item.getCategory().equals("ELECTRONIC")) {
                        total += item.getPrice() * 0.95;
                    } else {
                        total += item.getPrice();
                    }
                }
            }

            if (user.isPremium()) {
                total = total * 0.85;
            }

            if (total < 100) {
                total += 10;
            }

            return total * 1.08;
        }
    }
    return 0;
}
~~~

### Vì sao đây là code smell?

Method trên có nhiều dấu hiệu khó bảo trì:

- Có nhiều nhánh if lồng nhau.
- Một method xử lý nhiều trách nhiệm khác nhau.
- Logic tính giá, discount, shipping và tax bị trộn vào cùng một chỗ.
- Nếu muốn sửa một rule nhỏ, lập trình viên phải đọc toàn bộ method.
- Khi logic tăng thêm, method sẽ ngày càng dài và dễ gây bug.

Trong code smell detection, sample như trên có thể được gán nhãn **Positive** cho smell Complex Method.

### Một hướng refactor

Ta có thể tách method lớn thành nhiều method nhỏ hơn:

~~~java
public double calculateFinalPrice(User user, Cart cart) {
    if (!isValidOrder(user, cart)) {
        return 0;
    }

    double subtotal = calculateSubtotal(cart);
    double discounted = applyUserDiscount(user, subtotal);
    double withShipping = addShippingFee(discounted);
    return addTax(withShipping);
}
~~~

Sau refactor, method chính ngắn hơn và thể hiện rõ các bước xử lý. Mỗi phần logic có thể được kiểm thử riêng.

### Liên hệ với DeepSmells

Trong lab này, ta không viết rule thủ công như “đếm số if” hay “đếm số dòng”. Thay vào đó, DeepSmells học pattern từ dữ liệu đã tokenized.

Một sample code được biểu diễn thành chuỗi số:

~~~text
12 93 41 7 18 52 ...
~~~

Model nhận chuỗi số này và dự đoán:

- 1: sample có smell.
- 0: sample không có smell.

## Hình thức lab

Một số cell code có marker:

~~~python
# Your code in here
~~~

Sinh viên đọc phần mô tả trước mỗi bài tập, sau đó hoàn thiện phần code tương ứng. Các cell setup phụ trợ vẫn được cung cấp sẵn để lab tập trung vào data pipeline và model.

## Bối cảnh bài toán

Code smell là các dấu hiệu trong source code có thể làm giảm khả năng bảo trì, mở rộng hoặc tái sử dụng. Trong lab này, source code đã được tokenized thành chuỗi số nguyên. Mỗi dòng dữ liệu là một sample.

DeepSmells xử lý từng loại smell như một bài toán binary classification riêng:

- Input: một method/class đã được tokenized thành chuỗi số.
- Output: 1 nếu có smell, 0 nếu không có smell.


## Phần 0. Chuẩn Bị Dataset Trên Kaggle

### Mục tiêu

Đảm bảo Kaggle Notebook nhìn thấy thư mục dữ liệu tokenizer_cs.


In [1]:
from pathlib import Path
from dataclasses import dataclass
import os
import sys
import math
import random
import time
import datetime
import gc
import subprocess
import importlib.util

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')
WORKING_DIR = KAGGLE_WORKING if KAGGLE_WORKING.exists() else Path.cwd()

print('Python:', sys.version)
print('cwd:', Path.cwd())
print('/kaggle/input exists:', KAGGLE_INPUT.exists())
print('working dir:', WORKING_DIR)

Python: 3.12.12 (main, Dec 17 2025, 21:10:06) [Clang 21.1.4 ]
cwd: /home/nguyenquocdung/work/codesmell
/kaggle/input exists: False
working dir: /home/nguyenquocdung/work/codesmell


## Phần 1. Kiểm Tra Môi Trường Chạy

### Mục tiêu

Xác nhận notebook có đủ thư viện tối thiểu để chạy DeepSmells.

### Giải thích

DeepSmells chính trong lab này dùng PyTorch. Ta cần các thư viện:

- `numpy`: xử lý mảng dữ liệu token.
- `scikit-learn`: chia train/validation và tính metric.
- `torch`: xây dựng và huấn luyện model.
- `tqdm`: hiển thị progress bar.

Không dùng trực tiếp `requirements.txt` của repo gốc vì file đó pin nhiều wheel CUDA/TensorFlow cụ thể, có thể không phù hợp môi trường Kaggle.

### Checkpoint

Cell dưới phải in ra `All required dependencies are available` hoặc tự cài các thư viện còn thiếu.

In [2]:
required = {
    'numpy': 'numpy',
    'sklearn': 'scikit-learn',
    'torch': 'torch',
    'tqdm': 'tqdm',
}

missing = [pip_name for import_name, pip_name in required.items() if importlib.util.find_spec(import_name) is None]
print('Missing:', missing)
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
    print('Installed missing dependencies.')
else:
    print('All required dependencies are available.')

Missing: []
All required dependencies are available.


In [3]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import confusion_matrix
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

torch: 2.11.0+cu128
cuda available: True
gpu: NVIDIA GeForce RTX 5060 Ti


## Phần 2. Cấu Hình Thí Nghiệm

### Mục tiêu

Thiết lập smell, model và chế độ chạy trước khi load dữ liệu.

### Tham số sinh viên cần hiểu

- SMELL: loại code smell cần phát hiện. Mỗi smell là một bài toán binary classification riêng.
- DIM: dùng 1d vì model chính nhận chuỗi token một chiều.
- MODEL_NAME: chọn DeepSmells cho LSTM một chiều hoặc DeepSmells-BiLSTM cho LSTM hai chiều.
- FAST_DEV_RUN: chế độ chạy nhanh để kiểm tra pipeline trước khi train thật.
- NB_EPOCHS: số vòng huấn luyện.
- HIDDEN_SIZE_LSTM: số chiều hidden state của LSTM.
- MAX_EVAL_SAMPLES: giới hạn số sample eval để tránh quá tải RAM khi demo.

### Quy trình đề xuất

Lần chạy đầu tiên nên giữ FAST_DEV_RUN=True và RUN_TRAINING=False. Khi mọi checkpoint đều đúng, sinh viên mới bật training.

### Câu hỏi nhanh

- Nếu đổi từ DeepSmells sang DeepSmells-BiLSTM, hidden vector đưa vào classifier thay đổi thế nào?
- Vì sao ta không nên train full grid search ngay ở lần chạy đầu tiên?


In [ ]:
SMELL_NAMES = ['ComplexMethod', 'ComplexConditional', 'FeatureEnvy', 'MultifacetedAbstraction']

SMELL = 'ComplexMethod'
DIM = '1d'
MODEL_NAME = 'DeepSmells'  # 'DeepSmells' or 'DeepSmells-BiLSTM'

FAST_DEV_RUN = False
NB_EPOCHS = 15 if FAST_DEV_RUN else 60
TRAIN_BATCHSIZE = 128
VALID_BATCHSIZE = 128
LR = 0.03
THRESHOLD = 0.5
HIDDEN_SIZE_LSTM = 100

MAX_TRAINING_SAMPLES = 5000
MAX_EVAL_SAMPLES = 5000 if FAST_DEV_RUN else None

SEED = 0
CHECKPOINT_DIR = WORKING_DIR / 'deepsmells_checkpoints'
TRACKING_DIR = WORKING_DIR / 'deepsmells_tracking'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
TRACKING_DIR.mkdir(parents=True, exist_ok=True)

print('SMELL:', SMELL)
print('MODEL_NAME:', MODEL_NAME)
print('FAST_DEV_RUN:', FAST_DEV_RUN)
print('NB_EPOCHS:', NB_EPOCHS)
print('MAX_EVAL_SAMPLES:', MAX_EVAL_SAMPLES)


## Phần 3. Bài Tập 1 - Tự Động Tìm Thư Mục Dữ Liệu

### Mục tiêu

Hoàn thiện hàm tìm folder tokenizer_cs trong Kaggle input.

### Yêu cầu

Một folder được xem là tokenizer_cs hợp lệ nếu nó chứa cấu trúc dữ liệu cho ít nhất một smell, gồm nhánh 1d/Positive và 1d/Negative.

Sinh viên cần hoàn thiện hai hàm:

1. looks_like_tokenizer_cs(path): kiểm tra một path có phải root dữ liệu tokenized hay không.
2. find_tokenizer_cs(): tạo danh sách các vị trí có thể chứa dữ liệu, duyệt từng vị trí, và trả về vị trí hợp lệ đầu tiên.

### Kiến thức cần dùng

- Path.exists()
- Path.is_dir()
- Duyệt folder con bằng Path.iterdir()
- Ghép đường dẫn bằng toán tử /

### Checkpoint

Cell phải in được DATA_ROOT và danh sách smell tìm thấy.


In [ ]:
def looks_like_tokenizer_cs(path: Path) -> bool:
    """Return True if path looks like the tokenizer_cs root."""
    path = Path(path)
    if not path.exists() or not path.is_dir():
        return False

    for smell in SMELL_NAMES:
        positive_dir = path / smell / '1d' / 'Positive'
        negative_dir = path / smell / '1d' / 'Negative'

        if positive_dir.is_dir() and negative_dir.is_dir():
            return True

    return False


def find_tokenizer_cs() -> Path:
    """Find tokenizer_cs in current directory or /kaggle/input."""
    candidates = []
    cwd = Path.cwd()

    candidates.extend([
        cwd / 'tokenizer_cs',
        cwd / 'data' / 'tokenizer_cs',
        cwd / 'DeepSmells' / 'data' / 'tokenizer_cs',
    ])

    if KAGGLE_INPUT.exists():
        for dataset_dir in sorted(p for p in KAGGLE_INPUT.iterdir() if p.is_dir()):
            candidates.extend([
                dataset_dir / 'tokenizer_cs',
                dataset_dir / 'data' / 'tokenizer_cs',
                dataset_dir / 'DeepSmells' / 'data' / 'tokenizer_cs',
            ])

            for child in list(dataset_dir.iterdir())[:100]:
                if child.is_dir():
                    candidates.extend([
                        child / 'tokenizer_cs',
                        child / 'data' / 'tokenizer_cs',
                    ])

    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)

        if looks_like_tokenizer_cs(candidate):
            return candidate

    # Fallback: deep search anywhere under /kaggle/input (any nesting depth).
    if KAGGLE_INPUT.exists():
        for hit in KAGGLE_INPUT.rglob('tokenizer_cs'):
            if looks_like_tokenizer_cs(hit):
                return hit

    raise FileNotFoundError('Khong tim thay tokenizer_cs. Hay attach Kaggle Dataset chua data tokenized.')


DATA_ROOT = find_tokenizer_cs()
print('DATA_ROOT:', DATA_ROOT)
print('Available smells:', [p.name for p in DATA_ROOT.iterdir() if p.is_dir()])


## Phần 4. Bài Tập 2 - Khám Phá Dữ Liệu Raw

### Mục tiêu

Đếm số file và số sample raw trong từng smell/nhãn.

### Yêu cầu

Sinh viên cần hoàn thiện:

1. count_lines(folder): trả về tổng số dòng của tất cả file trong một folder.
2. Vòng lặp thống kê: duyệt qua từng smell, từng dimension 1d/2d, từng label Positive/Negative, sau đó thêm một dòng thống kê vào rows.

### Ý nghĩa kết quả

Bảng thống kê giúp sinh viên thấy dữ liệu có mất cân bằng hay không. Nếu Negative lớn hơn Positive rất nhiều, model dễ thiên về dự đoán Negative nếu không dùng weighted loss.

### Checkpoint

Bảng in ra có các cột: Smell, Dim, Label, Files, Raw lines.


In [6]:
def count_lines(folder: Path) -> int:
    """Count total lines across all files in a folder."""
    total = 0
    for file in folder.glob('*'):
        if not file.is_file():
            continue
        with file.open('r', errors='ignore') as f:
            for line in f:
                if line.strip():
                    total += 1
    return total


rows = []
for smell_dir in sorted(p for p in DATA_ROOT.iterdir() if p.is_dir()):
    for dim in ['1d', '2d']:
        for label in ['Positive', 'Negative']:
            folder = smell_dir / dim / label
            if not folder.exists():
                continue

            n_files = sum(1 for p in folder.glob('*') if p.is_file() and not p.name.startswith('.'))
            n_lines = count_lines(folder)
            rows.append((smell_dir.name, dim, label, n_files, n_lines))

print(f"{'Smell':28} {'Dim':4} {'Label':10} {'Files':>6} {'Raw lines':>10}")
for smell, dim, label, files, lines in rows:
    print(f"{smell:28} {dim:4} {label:10} {files:6} {lines:10}")


Smell                        Dim  Label       Files  Raw lines
ComplexConditional           1d   Positive        1       6523
ComplexConditional           1d   Negative        4     381016
ComplexConditional           2d   Positive        1     136954
ComplexConditional           2d   Negative        6    4159633
ComplexMethod                1d   Positive        1      26164
ComplexMethod                1d   Negative        5     466503
ComplexMethod                2d   Positive        1     733471
ComplexMethod                2d   Negative        5    3564781
FeatureEnvy                  1d   Positive        1       1918
FeatureEnvy                  1d   Negative       12     171300
FeatureEnvy                  2d   Positive        1     367993
FeatureEnvy                  2d   Negative        7    6124479
MultifacetedAbstraction      1d   Positive        1        307
MultifacetedAbstraction      1d   Negative       12     173757
MultifacetedAbstraction      2d   Positive        1    

## Phần 5. Bài Tập 3 - Xây Dựng Data Loader

### Mục tiêu

Hoàn thiện data loader để biến dữ liệu tokenized thành mảng NumPy có cùng độ dài.

### Pipeline cần cài đặt

1. Đọc từng dòng token id trong folder Positive và Negative.
2. Tính độ dài từng sample.
3. Tính max_input_length sau khi bỏ các sample quá dài bất thường.
4. Đọc lại sample hợp lệ và padding bằng 0 để mọi sample có cùng độ dài.
5. Tạo train/eval ban đầu, cân bằng số positive và negative trong train.
6. Gộp train/eval và chia lại bằng stratified split để tạo train/validation cuối cùng.

### Shape mong muốn

- train_data: N_train x max_input_length x 1
- train_labels: N_train
- valid_data: N_valid x max_input_length x 1
- valid_labels: N_valid

### Lưu ý triển khai

- Khi đọc token, mỗi dòng text cần được chuyển thành mảng số.
- Khi padding, chỉ sample có độ dài trong khoảng hợp lệ mới được giữ lại.
- Khi tạo label, Positive có nhãn 1.0 và Negative có nhãn 0.0.
- Khi chia train/validation cuối cùng, cần giữ tỉ lệ nhãn ổn định.

### Câu hỏi nhanh

- Vì sao model cần các sample trong cùng batch có cùng độ dài?
- Vì sao tập train được cân bằng nhưng validation vẫn có thể giữ mất cân bằng?


In [7]:
@dataclass
class InputData:
    train_data: np.ndarray
    train_labels: np.ndarray
    eval_data: np.ndarray
    eval_labels: np.ndarray
    max_input_length: int


def set_seed(seed: int = 0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def read_token_lengths(folder: Path, is_c2v: bool = False):
    """Return a list containing token length for each sample line."""
    lengths = []
    dtype = np.float32 if is_c2v else np.int32

    for file in folder.glob('*'):
        if file.name.startswith('.') or not file.is_file():
            continue
        with file.open('r', errors='ignore') as f:
            for line in f:
                text = line.replace('\t', ' ').strip()
                if not text:
                    continue

                arr = np.fromstring(text, dtype=dtype, sep=' ')
                lengths.append(len(arr))

    return lengths


def compute_max_without_upper_outliers(lengths, z: float = 1.0) -> int:
    """Compute max length after removing values greater than mean + z * std."""
    if not lengths:
        return 0

    values = np.asarray(lengths, dtype=np.float64)
    cutoff = values.mean() + z * values.std()

    kept = values[values <= cutoff]
    if kept.size == 0:
        return int(values.max())
    return int(kept.max())


def get_outlier_threshold(data_path: Path, z: float = 1.0, is_c2v: bool = False) -> int:
    """Compute shared max_input_length from Positive and Negative folders."""
    data_path = Path(data_path)
    pos_lengths = read_token_lengths(data_path / 'Positive', is_c2v=is_c2v)
    neg_lengths = read_token_lengths(data_path / 'Negative', is_c2v=is_c2v)

    pos_threshold = compute_max_without_upper_outliers(pos_lengths, z=z)
    neg_threshold = compute_max_without_upper_outliers(neg_lengths, z=z)

    return max(pos_threshold, neg_threshold)


def retrieve_data(folder: Path, max_len: int, is_c2v: bool = False):
    """Read tokenized samples, filter oversized samples, and zero-pad valid samples."""
    samples = []
    dtype = np.float32 if is_c2v else np.int32

    for file in folder.glob('*'):
        if file.name.startswith('.') or not file.is_file():
            continue
        with file.open('r', errors='ignore') as f:
            for line in f:
                text = line.replace('\t', ' ').strip()
                if not text:
                    continue

                arr = np.fromstring(text, dtype=dtype, sep=' ')
                arr_size = len(arr)
                if not (0 < arr_size <= max_len):
                    continue

                padded = np.zeros(max_len, dtype=np.float32)
                padded[:arr_size] = arr
                samples.append(padded)

    return samples


def tail(items, n):
    return [] if n <= 0 else items[-n:]


def get_data(data_path, train_validate_ratio=0.7, max_training_samples=5000, max_eval_samples=150000, is_c2v=False, seed=0):
    """Load Positive/Negative data and create initial train/eval arrays."""
    data_path = Path(data_path)
    rng = random.Random(seed)

    max_input_length = get_outlier_threshold(data_path, z=1, is_c2v=is_c2v)
    if max_input_length <= 0:
        raise ValueError(f'Khong tim thay sample hop le trong {data_path}')

    pos_data = retrieve_data(data_path / 'Positive', max_input_length, is_c2v=is_c2v)
    neg_data = retrieve_data(data_path / 'Negative', max_input_length, is_c2v=is_c2v)
    rng.shuffle(pos_data)
    rng.shuffle(neg_data)

    total_positive = len(pos_data)
    total_negative = len(neg_data)

    train_pos = int(train_validate_ratio * total_positive)
    eval_pos = total_positive - train_pos
    train_neg = int(train_validate_ratio * total_negative)
    eval_neg = total_negative - train_neg

    # Balance training samples and cap training size.
    train_pos = train_neg = min(max_training_samples, train_pos, train_neg)

    # Cap eval negative samples to keep memory reasonable.
    if max_eval_samples is not None and eval_neg > max_eval_samples:
        removed_ratio = (eval_neg - max_eval_samples) / eval_neg
        eval_pos = int(eval_pos - eval_pos * removed_ratio)
        eval_neg = max_eval_samples

    training_data = pos_data[:train_pos] + neg_data[:train_neg]
    training_labels = np.empty(len(training_data), dtype=np.float32)
    training_labels[:train_pos] = 1.0
    training_labels[train_pos:] = 0.0

    eval_data = tail(pos_data, eval_pos) + tail(neg_data, eval_neg)
    eval_labels = np.empty(len(eval_data), dtype=np.float32)
    eval_labels[:eval_pos] = 1.0
    eval_labels[eval_pos:] = 0.0

    # Stack lists into N x L x 1 float arrays.
    training_data = np.asarray(training_data, dtype=np.float32).reshape(len(training_data), max_input_length, 1)
    eval_data = np.asarray(eval_data, dtype=np.float32).reshape(len(eval_data), max_input_length, 1)

    train_perm = np.random.default_rng(seed).permutation(len(training_labels))
    eval_perm = np.random.default_rng(seed + 1).permutation(len(eval_labels))
    training_data, training_labels = training_data[train_perm], training_labels[train_perm]
    eval_data, eval_labels = eval_data[eval_perm], eval_labels[eval_perm]

    return training_data, training_labels, eval_data, eval_labels, max_input_length


def get_all_data(data_root, smell, dim='1d', train_validate_ratio=0.7, max_training_samples=5000, max_eval_samples=None, seed=0):
    """Load one smell dataset and return stratified train/validation data."""
    if max_eval_samples is None:
        max_eval_samples = 150000 if smell in ['ComplexConditional', 'ComplexMethod'] else 50000

    data_path = Path(data_root) / smell / dim
    train_data, train_labels, eval_data, eval_labels, max_input_length = get_data(
        data_path,
        train_validate_ratio=train_validate_ratio,
        max_training_samples=max_training_samples,
        max_eval_samples=max_eval_samples,
        seed=seed,
    )

    all_data = np.concatenate((train_data, eval_data), axis=0)
    all_labels = np.concatenate((train_labels, eval_labels), axis=0)

    train_data, eval_data, train_labels, eval_labels = train_test_split(
        all_data,
        all_labels,
        test_size=1 - train_validate_ratio,
        stratify=all_labels,
        random_state=seed,
    )

    return InputData(
        train_data=train_data,
        train_labels=train_labels,
        eval_data=eval_data,
        eval_labels=eval_labels,
        max_input_length=max_input_length,
    )


## Phần 6. Load Dữ Liệu Cho Một Smell

### Mục tiêu

Chạy data loader cho một smell cụ thể và kiểm tra shape dữ liệu.

### Việc cần quan sát

Sau cell dưới, sinh viên cần ghi nhận:

- `max_input_length`: độ dài chuỗi token sau khi loại outlier.
- `train_data.shape`: số sample train và chiều input.
- `valid_data.shape`: số sample validation và chiều input.
- `positive ratio`: tỉ lệ sample có smell.

### Checkpoint

Nếu cell này chạy thành công, notebook đã load đúng data từ Kaggle.

In [8]:
set_seed(SEED)

input_data = get_all_data(
    DATA_ROOT,
    smell=SMELL,
    dim=DIM,
    max_training_samples=MAX_TRAINING_SAMPLES,
    max_eval_samples=MAX_EVAL_SAMPLES,
    seed=SEED,
)

print('max_input_length:', input_data.max_input_length)
print('train_data:', input_data.train_data.shape)
print('train_labels:', input_data.train_labels.shape, 'positive ratio:', float(input_data.train_labels.mean()))
print('valid_data:', input_data.eval_data.shape)
print('valid_labels:', input_data.eval_labels.shape, 'positive ratio:', float(input_data.eval_labels.mean()))

max_input_length: 1071
train_data: (10687, 1071, 1)
train_labels: (10687,) positive ratio: 0.3449985980987549
valid_data: (4581, 1071, 1)
valid_labels: (4581,) positive ratio: 0.3451211452484131


## Phần 7. Bài Tập 4 - Tạo PyTorch Dataset Và DataLoader

### Mục tiêu

Hoàn thiện class CodeSmellDataset để PyTorch có thể tạo batch cho model.

### Yêu cầu

CodeSmellDataset cần làm ba việc:

1. Lưu input và label trong __init__.
2. Trong __getitem__, lấy một sample, reshape input thành 1 x L, chuyển input và label thành torch.Tensor kiểu float32.
3. Trong __len__, trả về số sample.

### Shape cần đạt

Conv1d yêu cầu input batch có dạng batch_size x channels x sequence_length. Vì chuỗi token chỉ có một kênh, mỗi sample cần có shape 1 x max_input_length.

### Checkpoint

Batch đầu tiên phải có shape batch_size x 1 x max_input_length, label có shape batch_size x 1.


In [9]:
class CodeSmellDataset(Dataset):
    def __init__(self, inputs, labels=None, is_test=False):
        self.inputs = inputs
        self.labels = labels
        self.is_test = is_test

    def __getitem__(self, idx):
        sample_input = torch.tensor(self.inputs[idx].reshape(1, -1), dtype=torch.float32)
        if self.is_test:
            return sample_input

        sample_label = torch.tensor([self.labels[idx]], dtype=torch.float32)
        return sample_input, sample_label

    def __len__(self):
        return len(self.inputs)


train_set = CodeSmellDataset(input_data.train_data, input_data.train_labels)
valid_set = CodeSmellDataset(input_data.eval_data, input_data.eval_labels)

train_loader = DataLoader(train_set, batch_size=TRAIN_BATCHSIZE, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=VALID_BATCHSIZE, shuffle=False)

batch_inputs, batch_labels = next(iter(train_loader))
print('batch_inputs:', tuple(batch_inputs.shape))
print('batch_labels:', tuple(batch_labels.shape))


batch_inputs: (128, 1, 1071)
batch_labels: (128, 1)


## Phần 8. Bài Tập 5 - Cài Đặt Mô Hình DeepSmells Nâng Cao Nhẹ

### Mục tiêu

Tự cài đặt kiến trúc DeepSmells bằng PyTorch, nhưng theo cách gọn và có tính tái sử dụng hơn.

### Điểm khó hơn so với bản cơ bản

Ở phần này, sinh viên không chỉ điền từng layer rời rạc. Các em cần tổ chức model thành các khối rõ ràng:

1. Viết một hàm tạo convolution block dùng lại cho cả hai model.
2. Cài đặt CNN_LSTM và CNN_BiLSTM dựa trên block đó.
3. Tự xử lý hidden state trả về từ LSTM và BiLSTM.
4. Thêm dropout vào classifier để giảm overfitting.
5. Tự tính input_size_lstm sau hai lần convolution + pooling.
6. Viết build_model để chọn đúng model theo tên.

### Kiến trúc cần đạt

Input của model có dạng:

~~~text
batch_size x 1 x sequence_length
~~~

Khối CNN gồm hai block liên tiếp:

~~~text
Conv1d -> BatchNorm1d -> ReLU -> MaxPool1d
Conv1d -> BatchNorm1d -> ReLU -> MaxPool1d
~~~

Sau CNN, tensor được đưa vào LSTM hoặc BiLSTM. Classifier cuối cùng nhận hidden state và trả về một logit cho mỗi sample.

### Yêu cầu cho CNN_LSTM

- Dùng LSTM một chiều.
- Hidden state cuối cùng có kích thước hidden_size_lstm.
- Classifier đầu vào là hidden_size_lstm.

### Yêu cầu cho CNN_BiLSTM

- Dùng LSTM hai chiều.
- Hidden state cuối cùng cần kết hợp hai hướng.
- Classifier đầu vào là hidden_size_lstm * 2.

### Yêu cầu cho classifier

Classifier gồm hai tầng ẩn, ReLU sau mỗi tầng ẩn, và dropout sau ReLU. Output cuối cùng là một logit, không dùng sigmoid trong forward.

### Checkpoint

Sau khi hoàn thiện, forward pass phải trả về logits shape batch_size x 1.


In [10]:
def make_conv_block(in_channels, out_channels, kernel_size):
    """Create one Conv1d -> BatchNorm1d -> ReLU -> MaxPool1d block."""
    layers = [
        nn.Conv1d(in_channels, out_channels, kernel_size),
        nn.BatchNorm1d(out_channels),
        nn.ReLU(),
        nn.MaxPool1d(kernel_size=2),
    ]
    return nn.Sequential(*layers)


class CNN_LSTM(nn.Module):
    def __init__(
        self,
        kernel_size,
        input_size_lstm,
        hidden_size_lstm,
        input_dim=1,
        conv_dim1=16,
        conv_dim2=32,
        hidden_fc1=64,
        hidden_fc2=64,
        num_classes=1,
        dropout=0.2,
    ):
        super().__init__()

        self.conv_layers = nn.Sequential(
            make_conv_block(input_dim, conv_dim1, kernel_size),
            make_conv_block(conv_dim1, conv_dim2, kernel_size),
        )

        self.lstm = nn.LSTM(
            input_size=input_size_lstm,
            hidden_size=hidden_size_lstm,
            batch_first=True,
        )

        self.dense_layers = nn.Sequential(
            nn.Linear(hidden_size_lstm, hidden_fc1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_fc1, hidden_fc2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_fc2, num_classes),
        )

    def forward(self, text):
        out = self.conv_layers(text)

        out, (h_n, c_n) = self.lstm(out)
        hidden = h_n[-1]
        logits = self.dense_layers(hidden)

        return logits


class CNN_BiLSTM(nn.Module):
    def __init__(
        self,
        kernel_size,
        input_size_lstm,
        hidden_size_lstm,
        input_dim=1,
        conv_dim1=16,
        conv_dim2=32,
        hidden_fc1=64,
        hidden_fc2=64,
        num_classes=1,
        dropout=0.2,
    ):
        super().__init__()
        self.hidden_size_lstm = hidden_size_lstm

        self.conv_layers = nn.Sequential(
            make_conv_block(input_dim, conv_dim1, kernel_size),
            make_conv_block(conv_dim1, conv_dim2, kernel_size),
        )

        self.lstm = nn.LSTM(
            input_size=input_size_lstm,
            hidden_size=hidden_size_lstm,
            batch_first=True,
            bidirectional=True,
        )

        self.dense_layers = nn.Sequential(
            nn.Linear(hidden_size_lstm * 2, hidden_fc1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_fc1, hidden_fc2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_fc2, num_classes),
        )

    def forward(self, text):
        out = self.conv_layers(text)

        out, (h_n, c_n) = self.lstm(out)
        hidden = torch.cat((h_n[-2], h_n[-1]), dim=1)
        logits = self.dense_layers(hidden)

        return logits


def size_output_conv(Lin, padding=0, dilation=1, kernel_size=3, stride=1):
    """Return output length of 1D convolution/pooling."""
    return (Lin + 2 * padding - dilation * (kernel_size - 1) - 1) // stride + 1


def calculate_size_lstm(input_size, kernel_size):
    """Return reduced sequence length after two Conv1d + MaxPool1d blocks."""
    out = input_size

    for _ in range(2):
        out = size_output_conv(out, kernel_size=kernel_size, stride=1)
        out = size_output_conv(out, kernel_size=2, stride=2)

    if out <= 0:
        raise ValueError(f'Invalid LSTM input size after CNN: {out}. Try smaller kernel_size.')
    return out


def build_model(model_name, kernel_size, length_code, hidden_size_lstm):
    """Build CNN_LSTM or CNN_BiLSTM and return model, input_size_lstm."""
    input_size_lstm = calculate_size_lstm(input_size=length_code, kernel_size=kernel_size)

    model_registry = {
        'DeepSmells': CNN_LSTM,
        'DeepSmells-BiLSTM': CNN_BiLSTM,
    }

    if model_name not in model_registry:
        raise ValueError(f'Unknown model_name: {model_name}')

    model = model_registry[model_name](
        kernel_size=kernel_size,
        input_size_lstm=input_size_lstm,
        hidden_size_lstm=hidden_size_lstm,
    )

    return model, input_size_lstm


## Phần 9. Bài Tập 6 - Kiểm Tra Forward Pass

### Mục tiêu

Tự viết đoạn kiểm tra model trên một batch thật.

### Yêu cầu

Sinh viên cần chọn device, tạo model với build_model, chuyển model và batch input sang device, chạy model trong chế độ không tính gradient, rồi in shape của logits.

### Checkpoint

logits shape phải là batch_size x 1.


In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
KERNEL_SIZE_SANITY = 5
length_code = batch_inputs.shape[-1]

model, input_size_lstm = build_model(MODEL_NAME, KERNEL_SIZE_SANITY, length_code, HIDDEN_SIZE_LSTM)
model = model.to(device)
model.eval()

with torch.no_grad():
    logits = model(batch_inputs.to(device))

print('device:', device)
print('length_code:', length_code)
print('input_size_lstm:', input_size_lstm)
print('logits shape:', tuple(logits.shape))
print('first logits:', logits[:5].detach().cpu().flatten().tolist())


device: cuda
length_code: 1071
input_size_lstm: 264
logits shape: (128, 1)
first logits: [0.08199959248304367, 0.09584983438253403, 0.02168736234307289, 0.06136970967054367, 0.02737792581319809]


## Phần 10. Bài Tập 7 - Training Loop Và Metric

### Mục tiêu

Hoàn thiện metric đánh giá và đọc hiểu training loop.

### Metric cần cài đặt

Hàm evaluation_metrics(y_pred_bool, y_true_bool) cần trả về bốn giá trị: Precision, Recall, F1 và MCC.

### Ý nghĩa metric

Precision cho biết trong các sample model dự đoán có smell, bao nhiêu sample thật sự có smell. Recall cho biết trong các sample thật sự có smell, model phát hiện được bao nhiêu. F1 cân bằng Precision và Recall. MCC hữu ích khi dữ liệu mất cân bằng mạnh.

### Training loop đã được cung cấp

Sinh viên chỉ cần hoàn thiện metric. Phần Trainer bên dưới dùng metric đó để train/validate.


In [12]:
def write_file(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a', encoding='utf-8') as f:
        f.write(text)

def evaluation_metrics(y_pred_bool, y_true_bool):
    precision = metrics.precision_score(y_true_bool, y_pred_bool, zero_division=0)
    recall = metrics.recall_score(y_true_bool, y_pred_bool, zero_division=0)
    f1 = metrics.f1_score(y_true_bool, y_pred_bool, zero_division=0)
    mcc = metrics.matthews_corrcoef(y_true_bool, y_pred_bool)

    return precision, recall, f1, mcc


class Trainer:
    def __init__(self, device, dataloader, model, loss_fns, optimizer, scheduler=None):
        self.device = device
        self.train_loader, self.valid_loader = dataloader
        self.model = model
        self.train_loss_fn, self.valid_loss_fn = loss_fns
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.scaler = GradScaler(enabled=(device.type == 'cuda'))
        self.threshold = 0.5
        self.train_loss = 0.0
        self.valid_loss = 0.0

    def train_one_epoch(self):
        self.model.train()
        train_preds, train_targets = [], []
        running_loss, seen = 0.0, 0
        pbar = tqdm(enumerate(self.train_loader), total=len(self.train_loader), desc='train')
        for _, (inputs_batch, targets) in pbar:
            inputs_batch = inputs_batch.to(self.device)
            targets = targets.to(self.device)

            self.optimizer.zero_grad()
            with autocast(enabled=(self.device.type == 'cuda')):
                outputs = self.model(inputs_batch)
                loss = self.train_loss_fn(outputs, targets)

            self.scaler.scale(loss).backward()
            self.scaler.step(self.optimizer)
            self.scaler.update()

            bs = inputs_batch.size(0)
            running_loss += loss.item() * bs
            seen += bs

            preds = (torch.sigmoid(outputs) >= self.threshold).float()
            train_preds.extend(preds.detach().cpu().numpy().flatten().tolist())
            train_targets.extend(targets.detach().cpu().numpy().flatten().tolist())
            pbar.set_postfix(loss=running_loss / max(seen, 1))

        self.train_loss = running_loss / max(seen, 1)
        return train_preds, train_targets

    @torch.no_grad()
    def valid_one_epoch(self):
        self.model.eval()
        valid_preds, valid_targets = [], []
        running_loss, seen = 0.0, 0
        pbar = tqdm(enumerate(self.valid_loader), total=len(self.valid_loader), desc='valid')
        for _, (inputs_batch, targets) in pbar:
            inputs_batch = inputs_batch.to(self.device)
            targets = targets.to(self.device)

            outputs = self.model(inputs_batch)
            loss = self.valid_loss_fn(outputs, targets)

            bs = inputs_batch.size(0)
            running_loss += loss.item() * bs
            seen += bs

            preds = (torch.sigmoid(outputs) >= self.threshold).float()
            valid_preds.extend(preds.detach().cpu().numpy().flatten().tolist())
            valid_targets.extend(targets.detach().cpu().numpy().flatten().tolist())

        self.valid_loss = running_loss / max(seen, 1)
        return valid_preds, valid_targets

    def save_model(self, epoch, path, name):
        path = Path(path)
        path.mkdir(parents=True, exist_ok=True)
        state = {
            'epoch': epoch,
            'state_dict': self.model.state_dict(),
            'optimizer': self.optimizer.state_dict(),
        }
        torch.save(state, path / f'{name}.pth')

    def fit(self, epochs, output_dir, track_dir, custom_name, threshold=0.5):
        best = {'precision': 0, 'recall': 0, 'f1': 0, 'mcc': 0}
        self.threshold = threshold
        track_file = Path(track_dir) / f'{custom_name}.txt'
        for epoch in range(epochs):
            header = f"{'='*25} Epoch: {epoch + 1} / {epochs} {'='*25}\n"
            print(header.strip())
            write_file(track_file, header)

            train_preds, train_targets = self.train_one_epoch()
            train_loss = self.train_loss
            train_precision, train_recall, train_f1, train_mcc = evaluation_metrics(train_preds, train_targets)

            valid_preds, valid_targets = self.valid_one_epoch()
            valid_loss = self.valid_loss
            valid_precision, valid_recall, valid_f1, valid_mcc = evaluation_metrics(valid_preds, valid_targets)

            if self.scheduler is not None:
                self.scheduler.step()

            msg = (
                f'Train loss: {train_loss:.4f} | P={train_precision:.4f} R={train_recall:.4f} F1={train_f1:.4f} MCC={train_mcc:.4f}\n'
                f'Valid loss: {valid_loss:.4f} | P={valid_precision:.4f} R={valid_recall:.4f} F1={valid_f1:.4f} MCC={valid_mcc:.4f}\n'
            )
            print(msg)
            write_file(track_file, msg)

            if valid_f1 > best['f1']:
                best = {'precision': valid_precision, 'recall': valid_recall, 'f1': valid_f1, 'mcc': valid_mcc}
                self.save_model(epoch + 1, output_dir, custom_name)

            gc.collect()

        summary = f"Best: P={best['precision']:.4f} R={best['recall']:.4f} F1={best['f1']:.4f} MCC={best['mcc']:.4f}\n"
        print(summary)
        write_file(track_file, summary)
        return best


## Phần 11. Thực Hành Training

### Mục tiêu

Chạy một thí nghiệm huấn luyện nhỏ hoặc grid search giống paper/codebase.

### Việc sinh viên cần làm

Trong cell dưới, sinh viên có thể điều chỉnh:

- RUN_TRAINING: đổi thành True để train.
- POS_WEIGHT_SET: danh sách trọng số class positive.
- KERNEL_SIZE_SET: danh sách kernel size cho CNN.

### Grid search trong DeepSmells

Paper/codebase thử:

~~~text
pos_weight = [1, 2, 4, 8, 12, 32, 84]
kernel_size = [3, 4, 5, 6, 7]
~~~

Khi FAST_DEV_RUN=True, notebook chỉ chạy một cấu hình để tiết kiệm thời gian.

### Câu hỏi nhanh

- Tăng pos_weight thường làm Recall tăng hay giảm? Vì sao?
- Kernel size ảnh hưởng gì tới pattern mà CNN nhìn thấy?


In [13]:
RUN_TRAINING = True  # Đổi thành True khi bạn muốn train trên Kaggle.

POS_WEIGHT_SET = [1.0] if FAST_DEV_RUN else [1.0, 2.0, 4.0, 8.0, 12.0, 32.0, 84.0]
KERNEL_SIZE_SET = [5] if FAST_DEV_RUN else [3, 4, 5, 6, 7]

def run_training_grid():
    results = []
    length_code = train_set[0][0].shape[1]
    now = datetime.datetime.now().strftime('%d%m%Y_%H%M')
    for pos_weight_value in POS_WEIGHT_SET:
        for kernel_size in KERNEL_SIZE_SET:
            print('\n' + '#' * 80)
            print(f'pos_weight={pos_weight_value}, kernel_size={kernel_size}')
            model, input_size_lstm = build_model(MODEL_NAME, kernel_size, length_code, HIDDEN_SIZE_LSTM)
            model = model.to(device)
            pos_weight = torch.tensor(pos_weight_value, dtype=torch.float32, device=device)
            loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
            optimizer = optim.SGD(model.parameters(), lr=LR)
            trainer = Trainer(
                device=device,
                dataloader=(train_loader, valid_loader),
                model=model,
                loss_fns=(loss_fn, loss_fn),
                optimizer=optimizer,
            )
            custom_name = f'Model_{SMELL}_{MODEL_NAME}_{now}_posweight_{pos_weight_value}_kernel_{kernel_size}'
            best = trainer.fit(
                epochs=NB_EPOCHS,
                output_dir=CHECKPOINT_DIR,
                track_dir=TRACKING_DIR,
                custom_name=custom_name,
                threshold=THRESHOLD,
            )
            result = {'smell': SMELL, 'model': MODEL_NAME, 'pos_weight': pos_weight_value, 'kernel_size': kernel_size, **best}
            results.append(result)
            del model, optimizer, trainer
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    return results

if RUN_TRAINING:
    train_results = run_training_grid()
    print(train_results)
else:
    print('RUN_TRAINING=False, skip training. Set RUN_TRAINING=True to train.')


################################################################################
pos_weight=1.0, kernel_size=5


========================= Epoch: 1 / 15 =========================


/tmp/ipykernel_18551/2792306491.py:24: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler(enabled=(device.type == 'cuda'))


train:   0%|          | 0/84 [00:00<?, ?it/s]

/tmp/ipykernel_18551/2792306491.py:39: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(self.device.type == 'cuda')):


...

train:   0%|          | 0/84 [00:00<?, ?it/s]

/tmp/ipykernel_18551/2792306491.py:39: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(self.device.type == 'cuda')):


valid:   0%|          | 0/36 [00:00<?, ?it/s]

Train loss: 0.3905 | P=0.8100 R=0.7448 F1=0.7760 MCC=0.6668
Valid loss: 0.3908 | P=0.7952 R=0.7489 F1=0.7713 MCC=0.6569

Best: P=0.7893 R=0.7628 F1=0.7758 MCC=0.6609



[{'smell': 'ComplexMethod', 'model': 'DeepSmells', 'pos_weight': 1.0, 'kernel_size': 5, 'precision': 0.7892670157068062, 'recall': 0.7628083491461101, 'f1': 0.7758121582502412, 'mcc': 0.6609332880568598}]


## Phần 12. Đọc Kết Quả Training

### Mục tiêu

Biết model checkpoint và log metric được lưu ở đâu.

### Output của lab

Sau training, notebook ghi:

- Checkpoint `.pth` trong `deepsmells_checkpoints`.
- Log `.txt` trong `deepsmells_tracking`.

Trên Kaggle, mọi file trong `/kaggle/working` sẽ xuất hiện ở phần Output sau khi notebook chạy xong.

### Bài tập nhỏ

Sau khi train ít nhất một cấu hình, hãy tìm cấu hình có validation F1 cao nhất trong log.

In [14]:
print('CHECKPOINT_DIR:', CHECKPOINT_DIR)
print('TRACKING_DIR:', TRACKING_DIR)
print('checkpoints:', [p.name for p in CHECKPOINT_DIR.glob('*.pth')][:10])
print('tracking logs:', [p.name for p in TRACKING_DIR.glob('*.txt')][:10])

for log_file in sorted(TRACKING_DIR.glob('*.txt'))[:3]:
    print('\n' + '=' * 80)
    print(log_file.name)
    print(log_file.read_text(errors='ignore')[-1200:])

CHECKPOINT_DIR: /home/nguyenquocdung/work/codesmell/deepsmells_checkpoints
TRACKING_DIR: /home/nguyenquocdung/work/codesmell/deepsmells_tracking
checkpoints: ['Model_ComplexMethod_DeepSmells_07062026_2334_posweight_1.0_kernel_5.pth']
tracking logs: ['Model_ComplexMethod_DeepSmells_07062026_2334_posweight_1.0_kernel_5.txt']

Model_ComplexMethod_DeepSmells_07062026_2334_posweight_1.0_kernel_5.txt
40 R=0.7318 F1=0.7662 MCC=0.6533
========================= Epoch: 10 / 15 =========================
Train loss: 0.3939 | P=0.8077 R=0.7350 F1=0.7697 MCC=0.6585
Valid loss: 0.3919 | P=0.8059 R=0.7249 F1=0.7632 MCC=0.6503
========================= Epoch: 11 / 15 =========================
Train loss: 0.3937 | P=0.8103 R=0.7394 F1=0.7732 MCC=0.6635
Valid loss: 0.3911 | P=0.8088 R=0.7198 F1=0.7617 MCC=0.6494
========================= Epoch: 12 / 15 =========================
Train loss: 0.3945 | P=0.8082 R=0.7337 F1=0.7691 MCC=0.6580
Valid loss: 0.3907 | P=0.8088 R=0.7198 F1=0.7617 MCC=0.6494
========

## Phần 13. Baseline Autoencoder

### Mục tiêu

Hiểu baseline mà DeepSmells được so sánh trong paper.

### Ý tưởng autoencoder

Autoencoder không trực tiếp học nhãn như classifier. Nó học tái tạo input:

```text
Input token vector -> Encoder -> Bottleneck -> Decoder -> Reconstructed input
```

Sau đó ta tính reconstruction error. Nếu error cao hơn threshold, sample có thể bị xem là smell.

### Các baseline trong notebook

- `AE-Dense`: encoder/decoder bằng dense layer.
- `AE-CNN`: encoder/decoder bằng convolution.
- `AE-LSTM`: encoder/decoder bằng LSTM.

### So sánh với DeepSmells

DeepSmells là supervised classifier: học trực tiếp từ label positive/negative. Autoencoder dựa trên reconstruction error và threshold. Vì vậy cách ra quyết định khác nhau.

### Lưu ý lab

Phần baseline mặc định tắt vì TensorFlow có thể tốn thời gian/RAM trên Kaggle. Sinh viên chỉ bật khi đã hoàn thành phần model chính.

In [15]:
RUN_AUTOENCODER_BASELINE = False

def ae_find_optimal(eval_data_2d, eval_labels, predictions, start=1000, stop=400000, step=5000):
    mse = np.mean(np.power(eval_data_2d - predictions, 2), axis=1)
    best = {'threshold': start, 'precision': 0, 'recall': 0, 'f1': 0, 'mcc': 0}
    for threshold in range(start, stop, step):
        y_pred = np.asarray([1 if e > threshold else 0 for e in mse])
        precision = metrics.precision_score(eval_labels, y_pred, zero_division=0)
        recall = metrics.recall_score(eval_labels, y_pred, zero_division=0)
        f1 = metrics.f1_score(eval_labels, y_pred, zero_division=0)
        mcc = metrics.matthews_corrcoef(eval_labels, y_pred)
        if f1 > best['f1']:
            best = {'threshold': threshold, 'precision': precision, 'recall': recall, 'f1': f1, 'mcc': mcc}
    return best

if RUN_AUTOENCODER_BASELINE:
    import tensorflow as tf
    from tensorflow.keras.layers import Input, Dense, LSTM, TimeDistributed, Conv1D, MaxPooling1D, UpSampling1D
    from tensorflow.keras.models import Model
    from tensorflow.python.keras import regularizers

    class AEInput:
        pass

    ae_data = AEInput()
    ae_data.train_data = input_data.train_data.reshape((input_data.train_data.shape[0], input_data.max_input_length))
    ae_data.eval_data = input_data.eval_data.reshape((input_data.eval_data.shape[0], input_data.max_input_length))
    ae_data.eval_labels = input_data.eval_labels
    ae_data.max_input_length = input_data.max_input_length

    def autoencoder_dense(data, layers=1, encoding_dimension=32, epochs=3, with_bottleneck=True):
        input_layer = Input(shape=(data.max_input_length,))
        prev = input_layer
        for i in range(layers):
            prev = Dense(int(encoding_dimension / pow(2, i)), activation='relu', activity_regularizer=regularizers.l1(1e-2))(prev)
        if with_bottleneck:
            prev = Dense(max(1, int(encoding_dimension / pow(2, layers))), activation='relu')(prev)
        for j in range(layers - 1, -1, -1):
            prev = Dense(int(encoding_dimension / pow(2, j)), activation='relu')(prev)
        output = Dense(data.max_input_length, activation='relu')(prev)
        model = Model(inputs=input_layer, outputs=output)
        model.compile(optimizer='adam', loss='mean_squared_error')
        model.fit(data.train_data, data.train_data, epochs=epochs, batch_size=128, validation_split=0.2, shuffle=True)
        predictions = model.predict(data.eval_data)
        return ae_find_optimal(data.eval_data, data.eval_labels, predictions)

    def align_prediction_width(predictions, target_width):
        if predictions.shape[1] > target_width:
            return predictions[:, :target_width]
        if predictions.shape[1] < target_width:
            pad_width = target_width - predictions.shape[1]
            return np.pad(predictions, ((0, 0), (0, pad_width)), mode='constant')
        return predictions

    def autoencoder_cnn(data, layers=1, filters=16, kernel=5, pooling_window=2, epochs=3):
        train_3d = data.train_data.reshape((len(data.train_data), data.max_input_length, 1))
        eval_3d = data.eval_data.reshape((len(data.eval_labels), data.max_input_length, 1))
        input_layer = Input(shape=(data.max_input_length, 1))
        prev = input_layer
        for i in range(layers):
            prev = Conv1D(max(1, int(filters / pow(2, i))), kernel, activation='relu', padding='same', kernel_initializer='random_uniform')(prev)
            prev = MaxPooling1D(pooling_window, strides=pooling_window)(prev)
        for j in range(layers - 1, -1, -1):
            prev = Conv1D(max(1, int(filters / pow(2, j))), kernel, activation='relu', padding='same', kernel_initializer='random_uniform')(prev)
            prev = UpSampling1D(pooling_window)(prev)
        output = Dense(1, activation='relu')(prev)
        model = Model(inputs=input_layer, outputs=output)
        model.compile(optimizer='adam', loss='mean_squared_error')
        model.fit(train_3d, train_3d, epochs=epochs, batch_size=128, validation_split=0.2, shuffle=True)
        predictions = model.predict(eval_3d).reshape((eval_3d.shape[0], -1))
        predictions = align_prediction_width(predictions, data.max_input_length)
        return ae_find_optimal(data.eval_data, data.eval_labels, predictions)

    def autoencoder_lstm(data, layers=1, encoding_dimension=8, epochs=3, with_bottleneck=True):
        train_3d = data.train_data.reshape((len(data.train_data), data.max_input_length, 1))
        eval_3d = data.eval_data.reshape((len(data.eval_labels), data.max_input_length, 1))
        input_layer = Input(shape=(data.max_input_length, 1))
        prev = input_layer
        for i in range(layers):
            prev = LSTM(max(1, int(encoding_dimension / pow(2, i))), return_sequences=True, dropout=0.1)(prev)
        if with_bottleneck:
            prev = LSTM(max(1, int(encoding_dimension / pow(2, layers + 1))), return_sequences=True, dropout=0.1)(prev)
        for j in range(layers - 1, -1, -1):
            prev = LSTM(max(1, int(encoding_dimension / pow(2, j))), return_sequences=True, dropout=0.1)(prev)
        output = TimeDistributed(Dense(1))(prev)
        model = Model(inputs=input_layer, outputs=output)
        model.compile(optimizer='adam', loss='mean_squared_error')
        model.fit(train_3d, train_3d, epochs=epochs, batch_size=64, validation_split=0.2, shuffle=True)
        predictions = model.predict(eval_3d).reshape((eval_3d.shape[0], -1))
        predictions = align_prediction_width(predictions, data.max_input_length)
        return ae_find_optimal(data.eval_data, data.eval_labels, predictions)

    print('AE-Dense:', autoencoder_dense(ae_data, epochs=1))
    # Uncomment if you want to run the heavier baselines too.
    # print('AE-CNN:', autoencoder_cnn(ae_data, epochs=1))
    # print('AE-LSTM:', autoencoder_lstm(ae_data, epochs=1))
else:
    print('RUN_AUTOENCODER_BASELINE=False, skip baseline.')

RUN_AUTOENCODER_BASELINE=False, skip baseline.


## Phần 14. Tổng Kết Lab Và Bài Nộp Gợi Ý

### Những gì sinh viên đã làm

Trong lab này, bạn đã tự hoàn thiện các phần quan trọng của DeepSmells:

1. Tìm data root trên Kaggle.
2. Thống kê dữ liệu raw.
3. Viết data loader với outlier filtering và zero padding.
4. Viết PyTorch Dataset/DataLoader.
5. Cài đặt model CNN-LSTM/BiLSTM.
6. Kiểm tra forward pass.
7. Cài đặt metric Precision, Recall, F1, MCC.
8. Chuẩn bị training loop và đọc log/checkpoint.

### Bài tập nộp sau lab

Sinh viên nộp báo cáo ngắn gồm:

1. Screenshot DATA_ROOT và bảng thống kê dữ liệu.
2. Screenshot shape của train_data, valid_data, batch input và label.
3. Screenshot forward pass với logits shape.
4. Nếu có train: bảng metric tốt nhất gồm Precision, Recall, F1, MCC.
5. Trả lời ngắn: CNN và LSTM đóng vai trò gì khác nhau trong DeepSmells?

### Gợi ý chạy trên Kaggle

- Bật GPU nếu train nhiều epoch.
- Chạy thử với FAST_DEV_RUN=True trước.
- Nếu hết RAM, giảm MAX_EVAL_SAMPLES hoặc batch size.
- Chạy từng smell một để dễ so sánh kết quả.
